# 14 — Real QES formalism + replica wormholes + higher-d RT surfaces (v2.0 patch)

**Spacetime Lab v2.0 — the sprint-closing major release** 🎉

Phase 9 (v1.0) and v1.1 identified the island *by hand* at the bifurcation surface. v2.0 implements the real formalism:

$$S_\mathrm{rad} = \min_X\;\mathrm{ext}_X\!\left[\frac{\mathrm{Area}(\partial X)}{4 G_N} + S_\mathrm{matter}(R \cup X)\right]$$

Three new modules:

1. `qes.py` — generic 1-parameter QES finder + JT dilaton + matter CFT toy model
2. `replica.py` — disconnected / connected replica saddles (reproducing Phase 9 bit-exactly)
3. `minimal_surfaces.py` — higher-d RT strip area in pure AdS, closed-form + numerical

This is the fifth and final patch of the v1.x post-roadmap sprint.

## 1. QES finder — JT + CFT bath toy model

Generalized entropy:
$$S_\mathrm{gen}(a) = \frac{\phi(a)}{4 G_N} + S_\mathrm{matter}([a, b])$$

with $\phi(a) = \phi_0 + \phi_r \tanh(\pi a/\beta)$ and Calabrese-Cardy matter entropy at finite $T = 1/\beta$.

The **extremality equation** $\partial S_\mathrm{gen}/\partial a = 0$ is solved numerically by `scipy.optimize.brentq` inside `find_qes`. No hand-identification.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from spacetime_lab.holography import (
    find_qes, jt_generalized_entropy, jt_generalized_entropy_derivative,
    island_formula_min, verify_qes_formalism,
    disconnected_saddle_entropy, connected_saddle_entropy,
    replica_island_saddle, verify_replica_at_n_equals_1,
    rt_strip_area_pure_ads, rt_strip_area_numerical,
    verify_rt_strip_against_closed_form,
)

# Default JT + bath parameters (chosen so a QES exists in bracket)
params = dict(phi_0=1.0, phi_r=10.0, beta=1.0, central_charge=1.0,
              b=2.0, epsilon=0.01, G_N=1.0)
qes = find_qes(**params)
print(f'QES located at a = {qes.a_qes:.6f}')
print(f'  S_gen at QES   = {qes.s_gen_at_qes:.6f}')
print(f'  residual       = {qes.residual:.2e}')
print(f'  a in (0, b)    = {0.0 < qes.a_qes < params["b"]}')

In [ ]:
# Plot S_gen(a) and its derivative
aas = np.linspace(0.01, params['b'] - 0.01, 300)
s_gen_vals = [jt_generalized_entropy(a, **{k: v for k, v in params.items() if k != 'epsilon' and k != 'b'}, b=params['b'], epsilon=params['epsilon']) for a in aas]
ds_vals = [jt_generalized_entropy_derivative(a, phi_r=params['phi_r'], beta=params['beta'], central_charge=params['central_charge'], b=params['b'], G_N=params['G_N']) for a in aas]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(aas, s_gen_vals, lw=2.2, color='#1f77b4')
axes[0].axvline(qes.a_qes, color='#d62728', ls=':', lw=1.5, label=f'QES a={qes.a_qes:.4f}')
axes[0].scatter([qes.a_qes], [qes.s_gen_at_qes], s=80, color='#d62728', zorder=5)
axes[0].set_xlabel('island boundary $a$')
axes[0].set_ylabel('$S_\\mathrm{gen}(a)$')
axes[0].set_title('Generalized entropy profile')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(aas, ds_vals, lw=2.2, color='#2ca02c')
axes[1].axhline(0, color='gray', ls='--', lw=1)
axes[1].axvline(qes.a_qes, color='#d62728', ls=':', lw=1.5)
axes[1].set_xlabel('island boundary $a$')
axes[1].set_ylabel('$\\partial S_\\mathrm{gen}/\\partial a$')
axes[1].set_title('Extremality equation (zero crossing = QES)')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Island formula min — full competition

The outer $\min$ of the island formula compares the no-island saddle to the island saddle found above. The winner is the physical entropy.

In [ ]:
d = island_formula_min(**params)
print('Island formula min:')
for k, v in d.items(): print(f'  {k}: {v}')
print()

# Scope note
print('Scope note: this static toy model has no time evolution, so the')
print('no-island saddle is time-independent and wins for the default parameters.')
print('The full Page-curve transition requires a time-dependent S_matter in')
print('the no-island saddle (radiation accumulation in the bath), which is')
print('handled in the replica-wormhole picture below and deferred to v2.1')
print('for the JT+bath dynamical wrapper.')

## 3. Replica wormholes — the dynamical Page transition

The replica wormhole calculation at $n = 1$ gives:

- **Disconnected saddle** $S_\mathrm{disc}(t) = \frac{2\pi c}{3\beta} t$ (Hawking linear growth, no island)
- **Connected saddle** $S_\mathrm{conn} = \pi r_+/G_N = 2 S_{BH}$ (island bound, time-independent)

The min of the two is the Page curve. Crossing at the **Page time** $t_P = 3\beta r_+ / (2 c G_N)$.

**Key v2.0 claim**: these two values match Phase 9's hand-identified saddles bit-exactly. `verify_replica_at_n_equals_1()` returns residuals of literally `0.0`.

In [ ]:
replica_diag = verify_replica_at_n_equals_1()
print('Replica vs Phase 9 cross-check:')
for k, v in replica_diag.items(): print(f'  {k}: {v}')

In [ ]:
# Plot the Page curve from the replica saddles
beta = 2 * math.pi
r_plus = 1.0
c = 6.0
ts = np.linspace(0.0, 4.0, 300)
s_disc = [disconnected_saddle_entropy(t, beta, c) for t in ts]
s_conn = [connected_saddle_entropy(r_plus) for _ in ts]
s_min  = [min(a, b_) for a, b_ in zip(s_disc, s_conn)]
t_P = 3 * beta * r_plus / (2 * c)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ts, s_disc, lw=1.5, color='#d62728', alpha=0.5, label='disconnected (Hawking)')
ax.plot(ts, s_conn, lw=1.5, color='#2ca02c', alpha=0.5, label='connected (replica wormhole)')
ax.plot(ts, s_min, lw=2.8, color='#1f1f1f', label='min = Page curve')
ax.axvline(t_P, color='gray', ls=':', lw=1.5, label=f'$t_P = 3\\beta r_+/(2 c G_N) = {t_P:.3f}$')
ax.set_xlabel('$t$'); ax.set_ylabel('entropy')
ax.set_title('Page curve from replica wormhole saddle competition')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Higher-dimensional RT strip surfaces

Numerical Ryu-Takayanagi strip area for pure AdS$_{d+1}$, verified against closed form across dimensions 2–6.

Closed form: $\mathcal{A} = \frac{2 L_{AdS}^{d-1}}{d-2}\left[\frac{1}{\epsilon^{d-2}} - \frac{C_d}{L^{d-2}}\right]$ for $d \ge 3$, and $\mathcal{A} = 2 L_{AdS} \log(L/\epsilon)$ for $d = 2$.

In [ ]:
verify_diag = verify_rt_strip_against_closed_form(L=2.0, epsilon=0.01, dimensions_to_check=(2, 3, 4, 5, 6))
print(f'{"d":>3}  {"closed":>18}  {"numerical":>18}  {"rel residual":>14}')
print('-' * 62)
for d, r in verify_diag.items():
    print(f'{d:>3}  {r["closed_form"]:>18.10g}  {r["numerical"]:>18.10g}  {r["rel_residual"]:>14.2e}')

In [ ]:
# Visualize the RT strip profile in AdS_4
d = 3
L_strip = 2.0
from math import gamma as _gamma
I_d = math.sqrt(math.pi) * _gamma(d/(2*(d-1))) / ((d-1) * _gamma(1/(2*(d-1))))
z_star = (L_strip/2) / I_d

# Solve for x(z) profile
zs = np.linspace(0.01, z_star * 0.999, 200)
xs = []
from scipy.integrate import quad
for z_end in zs:
    integrand = lambda z: 1.0 / math.sqrt((z_star/z)**(2*(d-1)) - 1)
    x_half, _ = quad(integrand, 0.001, z_end, limit=200)
    xs.append(x_half)
xs = np.array(xs)

# Mirror to get full surface
all_x = np.concatenate([-xs[::-1], xs])
all_z = np.concatenate([zs[::-1], zs])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(all_x, -all_z, lw=2.5, color='#1f77b4', label=f'RT surface (d={d+1}-dim bulk)')
ax.axhline(0, color='black', lw=2, label='boundary')
ax.axvspan(-L_strip/2, L_strip/2, ymin=0.95, alpha=0.4, color='#ff7f0e', label=f'boundary region (L={L_strip})')
ax.scatter([0], [-z_star], s=80, color='#d62728', zorder=5, label=f'turning point $z_*={z_star:.3f}$')
ax.set_xlabel('boundary $x_1$'); ax.set_ylabel('bulk $-z$')
ax.set_title(f'RT minimal surface profile in pure AdS$_{{{d+1}}}$ — strip geometry')
ax.set_xlim(-L_strip, L_strip); ax.set_ylim(-z_star * 1.1, 0.05)
ax.legend(loc='lower left'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Closing gate — all three modules pass


In [ ]:
# Gate 1: QES finder
qes_diag = verify_qes_formalism()
assert qes_diag['a_qes_in_bounds']
assert qes_diag['qes_residual'] < 1e-10
assert qes_diag['second_derivative_nonzero']

# Gate 2: Replica vs Phase 9 bit-exact
rep_diag = verify_replica_at_n_equals_1()
assert rep_diag['saddle_residual'] == 0.0
assert rep_diag['slope_residual'] == 0.0

# Gate 3: RT strip numerical vs closed form
rt_diag = verify_rt_strip_against_closed_form()
for d, r in rt_diag.items():
    assert r['rel_residual'] < 1e-3

print('✓ QES finder — residual {:.2e}, second-derivative nonzero'.format(qes_diag['qes_residual']))
print('✓ Replica saddle matches Phase 9 bit-exactly (residual 0.0)')
print('✓ Replica slope matches Phase 9 HM rate bit-exactly (residual 0.0)')
print('✓ RT strip d=2–6 all under 1e-3 relative residual')
print()
print('ALL V2.0 GATES PASS — sprint closure ready to ship.')

## 6. Sprint closure — the 5 post-v1.0 patches

v1.0 (2026-04-11) closed the 18-month Spacetime Lab roadmap at **Phase 9: island formula + Page curve**.  The v1.x sprint extended it with five patches:

| Patch | Release | Tests | Physics |
|---|---|---|---|
| v1.1 | 2026-04-11 | 472 → 509 | Evaporating Page curve (bell-shaped, unitary endpoint) |
| v1.2 | 2026-04-12 | 509 → 535 | Kerr QNM wrapper (m-splitting, BH spectroscopy) |
| v1.3 | 2026-04-12 | 535 → 569 | Rotating BTZ (ergoregion, rotating Strominger-Cardy) |
| v1.5 | 2026-04-12 | 569 → 598 | Pure-string SVG + TikZ Penrose renderers |
| **v2.0** | **2026-04-12** | **598 → 634** | **Real QES finder + replica wormholes + higher-d RT** |

### What v2.0 ships in one line

The island formula is no longer a hand-identified saddle — it's a computed extremum, and the answer agrees bit-exactly with the replica wormhole picture (`residual = 0.0`) on one side and reproduces all published closed forms for RT strip areas in higher dimensions on the other.

### Deferred to v2.1 and beyond (honest)

- **Full JT+bath dynamical wrapper** — time-dependent matter entropy producing Page curve from QES competition directly (currently replica gives dynamics, QES is static)
- **Two-parameter QES** — two-sided eternal BH with (a+, a-) extremality
- **Replica wormhole Euclidean path integral** — we ship the on-shell actions, not the path integral derivation
- **Black-hole RT backgrounds in higher d** — Schwarzschild-AdS and higher-d BTZ RT surfaces; v2.0 module handles pure AdS only

The five-patch sprint closes with v2.0 — Spacetime Lab has a real, non-stub implementation of every piece that was in the 18-month roadmap, plus the v1.x extensions.  🎉